CELL 1 — Install ALL required packages

In [7]:
# ============================================================
# CELL 1 — INSTALL ALL REQUIRED DEPENDENCIES
# ============================================================

%pip install -q --upgrade pip

%pip install -q \
    pymysql \
    pandas \
    numpy \
    matplotlib \
    scikit-learn \
    tqdm \
    selfies \
    rdkit \
    torch \
    torch-geometric

print("Package installation complete. Proceed to Cell 2.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Package installation complete. Proceed to Cell 2.


CELL 2 — Phase 1 Implementation

In [9]:
# ============================================================
# CELL 2 — PHASE 1: DRUGBANK MYSQL EXTRACTION & ROBUST 3D GRAPHS
# ============================================================

import os
import sys
import random
import warnings
import pymysql
from pymysql.cursors import DictCursor

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch_geometric.data import Data

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    AllChem,
    Descriptors,
    Lipinski,
    rdMolDescriptors,
    rdFingerprintGenerator
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Reproducibility & Device Configuration
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Operating Device:", DEVICE)

# ------------------------------------------------------------
# 2. Local XAMPP MySQL Database Configuration
# ------------------------------------------------------------
DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "",
    "database": "drugbank",
    "charset": "utf8mb4",
    "cursorclass": DictCursor
}

MAX_MOLECULES = 600      # Working corpus limit
MAX_ATOMS = 100          # Upper ceiling for atom count
FP_DIM = 256            # Morgan fingerprint dimension

# ------------------------------------------------------------
# 3. Database Extraction (D_chem and D_val)
# ------------------------------------------------------------
try:
    connection = pymysql.connect(**DB_CONFIG)
    print("Connected successfully to MySQL database:", DB_CONFIG["database"])
except Exception as exc:
    raise RuntimeError(
        f"\nCould not connect to MySQL on localhost.\n"
        f"Ensure XAMPP MySQL is running on port 3306 and database 'drugbank' exists.\n"
        f"Original error: {exc}"
    )

SMILES_QUERY = """
SELECT DISTINCT
    d.drug_pk,
    d.primary_drugbank_id AS drug_id,
    d.name AS drug_name,
    dp.value AS smiles
FROM drug AS d
INNER JOIN drug_property AS dp
    ON d.drug_pk = dp.drug_pk
WHERE dp.kind = 'SMILES'
  AND dp.value IS NOT NULL
  AND TRIM(dp.value) <> ''
LIMIT %s;
"""

TARGET_QUERY = """
SELECT DISTINCT
    d.drug_pk,
    d.primary_drugbank_id AS drug_id,
    d.name AS drug_name,
    dp.value AS smiles,
    i.interactant_pk,
    i.interactant_id,
    i.name AS target_name,
    i.organism AS target_organism,
    i.known_action,
    ip.polypeptide_id,
    ip.source AS polypeptide_source,
    p.amino_acid_sequence
FROM drug AS d
INNER JOIN drug_property AS dp
    ON d.drug_pk = dp.drug_pk
   AND dp.kind = 'SMILES'
INNER JOIN interactant AS i
    ON d.drug_pk = i.drug_pk
   AND i.kind = 'target'
INNER JOIN interactant_polypeptide AS ip
    ON i.interactant_pk = ip.interactant_pk
INNER JOIN polypeptide AS p
    ON ip.polypeptide_id = p.polypeptide_id
   AND ip.source = p.source
WHERE dp.value IS NOT NULL
  AND TRIM(dp.value) <> ''
  AND p.amino_acid_sequence IS NOT NULL
LIMIT %s;
"""

try:
    chem_df = pd.read_sql(SMILES_QUERY, connection, params=(MAX_MOLECULES,))
    target_df = pd.read_sql(TARGET_QUERY, connection, params=(MAX_MOLECULES,))
finally:
    connection.close()

print(f"Extracted chemical records (D_chem): {len(chem_df)}")
print(f"Extracted drug-target bioactivity records (D_val): {len(target_df)}")

if len(chem_df) == 0:
    raise ValueError("Query returned 0 rows. Verify that the table 'drug_property' contains kind='SMILES' entries.")

# ------------------------------------------------------------
# 4. SMILES Sanitization & Canonicalization
# ------------------------------------------------------------
def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

chem_df["smiles"] = chem_df["smiles"].apply(canonicalize_smiles)
chem_df = chem_df.dropna(subset=["smiles"]).copy()

# Discard multi-component salts/solvents for single organic structures
chem_df = chem_df[~chem_df["smiles"].str.contains(r"\.", regex=True)].copy()
chem_df = chem_df.drop_duplicates(subset=["drug_id"]).reset_index(drop=True)
print(f"Validated single-component structures: {len(chem_df)}")

# ------------------------------------------------------------
# 5. Featurization & Graph Builder G = (V, E, X, R)
# ------------------------------------------------------------
FP_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_DIM)

def get_fingerprint_array(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = FP_GENERATOR.GetFingerprint(mol)
    arr = np.zeros((FP_DIM,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def atom_features(atom):
    hybridizations = [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3
    ]
    hyb_onehot = [float(atom.GetHybridization() == h) for h in hybridizations]
    hyb_onehot.append(float(atom.GetHybridization() not in hybridizations))

    chiral_tags = [
        Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW
    ]
    chiral_onehot = [float(atom.GetChiralTag() == tag) for tag in chiral_tags]
    chiral_onehot.append(float(atom.GetChiralTag() not in chiral_tags))

    return [
        atom.GetAtomicNum() / 100.0,
        atom.GetFormalCharge() / 5.0,
        atom.GetDegree() / 6.0,
        float(atom.GetIsAromatic()),
        *hyb_onehot,
        *chiral_onehot
    ]

def bond_features(bond):
    bt = bond.GetBondType()
    return [
        float(bt == Chem.rdchem.BondType.SINGLE),
        float(bt == Chem.rdchem.BondType.DOUBLE),
        float(bt == Chem.rdchem.BondType.TRIPLE),
        float(bt == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated())
    ]

def smiles_to_spatial_graph(smiles, drug_id="UNKNOWN", drug_name="UNKNOWN"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumHeavyAtoms() == 0 or mol.GetNumHeavyAtoms() > MAX_ATOMS:
        return None

    # Step A: Generate 3D Conformer with Multi-Tier Fallback
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = SEED
    params.maxAttempts = 50

    # Attempt 1: Standard ETKDGv3
    embed_ok = (AllChem.EmbedMolecule(mol_h, params) == 0)

    # Attempt 2: Random coordinates fallback
    if not embed_ok:
        params.useRandomCoords = True
        embed_ok = (AllChem.EmbedMolecule(mol_h, params) == 0)

    # Attempt 3: 2D planar coordinates embedded into 3D (Z=0) if 3D fails
    if not embed_ok:
        AllChem.Compute2DCoords(mol)
        conf = mol.GetConformer()
        coords = [list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())]
    else:
        # Energy minimization on 3D conformer
        try:
            if AllChem.MMFFHasAllMoleculeParams(mol_h):
                AllChem.MMFFOptimizeMolecule(mol_h, maxIters=50)
            else:
                AllChem.UFFOptimizeMolecule(mol_h, maxIters=50)
            mol = Chem.RemoveHs(mol_h)
        except Exception:
            mol = Chem.RemoveHs(mol_h)

        if mol.GetNumConformers() == 0:
            AllChem.Compute2DCoords(mol)
        conf = mol.GetConformer()
        coords = [list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())]

    # Node attributes (X)
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)

    # 3D Cartesian coordinates (R) centered at origin
    pos = torch.tensor(coords, dtype=torch.float32)
    pos = pos - pos.mean(dim=0, keepdim=True)

    # Directed edges & attributes (E)
    src, dst, edge_attr = [], [], []
    for bond in mol.GetBonds():
        u, v = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feats = bond_features(bond)
        src.extend([u, v])
        dst.extend([v, u])
        edge_attr.extend([feats, feats])

    if len(src) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr_tensor = torch.empty((0, 5), dtype=torch.float32)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr_tensor = torch.tensor(edge_attr, dtype=torch.float32)

    fp = get_fingerprint_array(smiles)
    if fp is None:
        return None

    graph = Data(x=x, pos=pos, edge_index=edge_index, edge_attr=edge_attr_tensor)
    graph.fp = torch.tensor(fp, dtype=torch.float32).unsqueeze(0)
    graph.smiles = smiles
    graph.drug_id = str(drug_id)
    graph.drug_name = str(drug_name)
    return graph

# ------------------------------------------------------------
# 6. Build Graph Corpus
# ------------------------------------------------------------
baseline_graphs = []
for row in tqdm(chem_df.itertuples(index=False), total=len(chem_df), desc="Constructing 3D Attributed Graphs"):
    g = smiles_to_spatial_graph(row.smiles, drug_id=row.drug_id, drug_name=row.drug_name)
    if g is not None:
        baseline_graphs.append(g)

print(f"\nPhase 1 Complete: {len(baseline_graphs)} spatial graphs successfully constructed.")

if len(baseline_graphs) == 0:
    raise RuntimeError("Zero graphs were generated. Check your MySQL SMILES data and RDKit sanitization.")

NODE_FEATURE_DIM = baseline_graphs[0].x.shape[1]
EDGE_FEATURE_DIM = baseline_graphs[0].edge_attr.shape[1]
print(f"Graph Dimensions -> Node Feature Dim (X): {NODE_FEATURE_DIM}, Bond Feature Dim (E): {EDGE_FEATURE_DIM}")

# Display extracted dataframes preview
display(chem_df.head(3))
if len(target_df) > 0:
    display(target_df[["drug_id", "drug_name", "interactant_id", "target_name"]].head(3))

Operating Device: cpu
Connected successfully to MySQL database: drugbank
Extracted chemical records (D_chem): 600
Extracted drug-target bioactivity records (D_val): 600
Validated single-component structures: 0


[19:48:29] SMILES Parse Error: syntax error while parsing: smiles
[19:48:29] SMILES Parse Error: check for mistakes around position 2:
[19:48:29] smiles
[19:48:29] ~^
[19:48:29] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[19:48:29] SMILES Parse Error: syntax error while parsing: smiles
[19:48:29] SMILES Parse Error: check for mistakes around position 2:
[19:48:29] smiles
[19:48:29] ~^
[19:48:29] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[19:48:29] SMILES Parse Error: syntax error while parsing: smiles
[19:48:29] SMILES Parse Error: check for mistakes around position 2:
[19:48:29] smiles
[19:48:29] ~^
[19:48:29] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[19:48:29] SMILES Parse Error: syntax error while parsing: smiles
[19:48:29] SMILES Parse Error: check for mistakes around position 2:
[19:48:29] smiles
[19:48:29] ~^
[19:48:29] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[19:

Constructing 3D Attributed Graphs: 0it [00:00, ?it/s]


Phase 1 Complete: 0 spatial graphs successfully constructed.


RuntimeError: Zero graphs were generated. Check your MySQL SMILES data and RDKit sanitization.

CELL 3 — Connect to DrugBank and retrieve structures/targets

The query below follows your actual schema. interactant.kind='target' is supported by the metadata. Protein sequences are stored in polypeptide.amino_acid_sequence.

In [3]:
# ============================================================
# CELL 3 — MYSQL CONNECTION AND DRUGBANK EXTRACTION
# ============================================================

try:
    connection = pymysql.connect(**DB_CONFIG)
    print("Connected successfully to MySQL database:", DB_CONFIG["database"])

except Exception as exc:
    raise RuntimeError(
        "\nCould not connect to MySQL.\n"
        "Check that XAMPP MySQL is running and that the database "
        "'drugbank' exists.\n\n"
        f"Original error: {exc}"
    )


# ------------------------------------------------------------
# Query 1:
# Baseline chemical corpus D_chem
# ------------------------------------------------------------

SMILES_QUERY = """
SELECT DISTINCT
    d.drug_pk,
    d.primary_drugbank_id AS drug_id,
    d.name AS drug_name,
    dp.value AS smiles
FROM drug AS d
INNER JOIN drug_property AS dp
    ON d.drug_pk = dp.drug_pk
WHERE dp.kind = 'SMILES'
  AND dp.value IS NOT NULL
  AND TRIM(dp.value) <> ''
LIMIT %s;
"""


# ------------------------------------------------------------
# Query 2:
# Drug-target / polypeptide information
#
# IMPORTANT:
# ip.source = p.source is included because the schema uses
# (polypeptide_id, source) as the polypeptide key.
# ------------------------------------------------------------

TARGET_QUERY = """
SELECT DISTINCT
    d.drug_pk,
    d.primary_drugbank_id AS drug_id,
    d.name AS drug_name,
    dp.value AS smiles,
    i.interactant_pk,
    i.interactant_id,
    i.name AS target_name,
    i.organism AS target_organism,
    i.known_action,
    ip.polypeptide_id,
    ip.source AS polypeptide_source,
    p.amino_acid_sequence
FROM drug AS d
INNER JOIN drug_property AS dp
    ON d.drug_pk = dp.drug_pk
   AND dp.kind = 'SMILES'
INNER JOIN interactant AS i
    ON d.drug_pk = i.drug_pk
   AND i.kind = 'target'
INNER JOIN interactant_polypeptide AS ip
    ON i.interactant_pk = ip.interactant_pk
INNER JOIN polypeptide AS p
    ON ip.polypeptide_id = p.polypeptide_id
   AND ip.source = p.source
WHERE dp.value IS NOT NULL
  AND TRIM(dp.value) <> ''
  AND p.amino_acid_sequence IS NOT NULL;
"""


with connection.cursor() as cursor:

    cursor.execute(SMILES_QUERY, (MAX_MOLECULES,))
    chem_rows = cursor.fetchall()

    cursor.execute(TARGET_QUERY)
    target_rows = cursor.fetchall()


connection.close()


chem_df = pd.DataFrame(chem_rows)
target_df = pd.DataFrame(target_rows)

print("\nChemical records extracted:", len(chem_df))
print("Drug-target records extracted:", len(target_df))

display(chem_df.head())

if len(target_df) > 0:
    display(
        target_df[
            [
                "drug_id",
                "drug_name",
                "interactant_id",
                "target_name",
                "polypeptide_id"
            ]
        ].head()
    )

Connected successfully to MySQL database: drugbank

Chemical records extracted: 600
Drug-target records extracted: 24525


,drug_pk,drug_id,drug_name,smiles
0,294750,DB01022,Phylloquinone,CC(C)CCC[C@@H](C)CCC[C@@H](C)CCC\C(C)=C\CC1=C(...
1,294751,DB01373,Calcium,[Ca]
2,294758,DB00006,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...
3,294775,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
4,294790,DB00027,Gramicidin D,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...


,drug_id,drug_name,interactant_id,target_name,polypeptide_id
0,DB01022,Phylloquinone,BE0000403,Vitamin K-dependent gamma-carboxylase,P38435
1,DB01022,Phylloquinone,BE0000724,Osteocalcin,P02818
2,DB01373,Calcium,BE0000430,Voltage-dependent L-type calcium channel subun...,Q13936
3,DB01373,Calcium,BE0000218,Calcium-transporting ATPase type 2C member 1,P98194
4,DB01373,Calcium,BE0003566,"Troponin C, skeletal muscle",P02585


CELL 4 — Molecular preprocessing, RDKit 3-D graph generation and SA score

This creates:

$$ G=(V,E,X,R) $$

where:

\(V\) = atoms
\(E\) = bonds
\(X\) = atom features
\(R\) = Cartesian coordinates

In [4]:
# ============================================================
# CELL 4 — RDKIT CHEMISTRY + 3-D MOLECULAR GRAPH CONSTRUCTION
# ============================================================

# ------------------------------------------------------------
# RDKit Synthetic Accessibility scorer
# ------------------------------------------------------------

SA_SCORER_AVAILABLE = False

try:
    sa_path = os.path.join(RDConfig.RDContribDir, "SA_Score")

    if sa_path not in sys.path:
        sys.path.append(sa_path)

    import sascorer

    SA_SCORER_AVAILABLE = True
    print("RDKit fragment-based SA scorer loaded.")

except Exception as exc:
    print(
        "WARNING: RDKit contrib SA scorer was not located.\n"
        "A transparent heuristic fallback will be used instead."
    )
    print("Details:", exc)


# ------------------------------------------------------------
# Canonical SMILES
# ------------------------------------------------------------

def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return None

        return Chem.MolToSmiles(mol, canonical=True)

    except Exception:
        return None


chem_df["smiles"] = chem_df["smiles"].apply(canonicalize_smiles)

chem_df = chem_df.dropna(subset=["smiles"]).copy()

# Keep one-component structures for this lightweight POC.
chem_df = chem_df[
    ~chem_df["smiles"].str.contains(r"\.", regex=True)
].copy()

chem_df = chem_df.drop_duplicates(
    subset=["drug_id"]
).reset_index(drop=True)

print("Canonical single-component molecules:", len(chem_df))


# ------------------------------------------------------------
# Molecular fingerprint
# ------------------------------------------------------------

FP_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=FP_DIM
)


def get_rdkit_fp(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return FP_GENERATOR.GetFingerprint(mol)


def fingerprint_numpy(smiles):

    fp = get_rdkit_fp(smiles)

    if fp is None:
        return None

    array = np.zeros((FP_DIM,), dtype=np.float32)

    DataStructs.ConvertToNumpyArray(fp, array)

    return array


# ------------------------------------------------------------
# Synthetic Accessibility
# ------------------------------------------------------------

def calculate_sa(mol):

    if mol is None:
        return np.nan

    if SA_SCORER_AVAILABLE:
        try:
            return float(sascorer.calculateScore(mol))
        except Exception:
            pass

    # FALLBACK ONLY:
    # This is not the Ertl fragment SA score.
    mw = Descriptors.MolWt(mol)
    rot = Lipinski.NumRotatableBonds(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    stereo = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))

    value = (
        1.0
        + 0.003 * max(mw - 150.0, 0.0)
        + 0.20 * rot
        + 0.10 * rings
        + 0.10 * stereo
    )

    return float(np.clip(value, 1.0, 10.0))


# ------------------------------------------------------------
# Atom attributes X
# ------------------------------------------------------------

def atom_features(atom):

    hybridizations = [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3
    ]

    hybridization_features = [
        float(atom.GetHybridization() == h)
        for h in hybridizations
    ]

    hybridization_features.append(
        float(atom.GetHybridization() not in hybridizations)
    )

    chiral_tags = [
        Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW
    ]

    chirality_features = [
        float(atom.GetChiralTag() == tag)
        for tag in chiral_tags
    ]

    chirality_features.append(
        float(atom.GetChiralTag() not in chiral_tags)
    )

    return [
        atom.GetAtomicNum() / 100.0,
        atom.GetFormalCharge() / 5.0,
        atom.GetDegree() / 6.0,
        float(atom.GetIsAromatic()),
        *hybridization_features,
        *chirality_features
    ]


# ------------------------------------------------------------
# Bond attributes E
# ------------------------------------------------------------

def bond_features(bond):

    bt = bond.GetBondType()

    return [
        float(bt == Chem.rdchem.BondType.SINGLE),
        float(bt == Chem.rdchem.BondType.DOUBLE),
        float(bt == Chem.rdchem.BondType.TRIPLE),
        float(bt == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated())
    ]


# ------------------------------------------------------------
# Generate attributed 3-D spatial graph
# ------------------------------------------------------------

def smiles_to_graph(smiles, drug_id="UNKNOWN", drug_name="UNKNOWN"):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    if mol.GetNumHeavyAtoms() > MAX_ATOMS:
        return None

    # Explicit hydrogen model for conformer generation
    mol_h = Chem.AddHs(mol)

    try:
        params = AllChem.ETKDGv3()
        params.randomSeed = SEED

        status = AllChem.EmbedMolecule(mol_h, params)

        # Retry using random coordinates if normal ETKDG fails
        if status != 0:

            params.useRandomCoords = True

            status = AllChem.EmbedMolecule(mol_h, params)

        if status != 0:
            return None

        # Energy minimization
        if AllChem.MMFFHasAllMoleculeParams(mol_h):

            AllChem.MMFFOptimizeMolecule(
                mol_h,
                maxIters=100
            )

        else:

            AllChem.UFFOptimizeMolecule(
                mol_h,
                maxIters=100
            )

        # Return to heavy-atom representation
        mol = Chem.RemoveHs(mol_h)

    except Exception:
        return None

    if mol.GetNumConformers() == 0:
        return None

    conformer = mol.GetConformer()

    # --------------------------------------------------------
    # Node features X
    # --------------------------------------------------------

    x = torch.tensor(
        [atom_features(a) for a in mol.GetAtoms()],
        dtype=torch.float32
    )

    # --------------------------------------------------------
    # Cartesian coordinates R
    # --------------------------------------------------------

    coordinates = []

    for index in range(mol.GetNumAtoms()):

        p = conformer.GetAtomPosition(index)

        coordinates.append(
            [p.x, p.y, p.z]
        )

    pos = torch.tensor(
        coordinates,
        dtype=torch.float32
    )

    # Centre coordinates.
    # Translation does not carry chemical information.
    pos = pos - pos.mean(dim=0, keepdim=True)

    # --------------------------------------------------------
    # Bidirectional graph edges and edge features
    # --------------------------------------------------------

    src = []
    dst = []
    edge_attr = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        features = bond_features(bond)

        src.extend([i, j])
        dst.extend([j, i])

        edge_attr.extend(
            [features, features]
        )

    if len(src) == 0:

        edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

        edge_attr_tensor = torch.empty(
            (0, 5),
            dtype=torch.float32
        )

    else:

        edge_index = torch.tensor(
            [src, dst],
            dtype=torch.long
        )

        edge_attr_tensor = torch.tensor(
            edge_attr,
            dtype=torch.float32
        )

    fp = fingerprint_numpy(smiles)

    if fp is None:
        return None

    graph = Data(
        x=x,
        pos=pos,
        edge_index=edge_index,
        edge_attr=edge_attr_tensor
    )

    # Shape [1, FP_DIM] allows PyG to form [B, FP_DIM]
    graph.fp = torch.tensor(
        fp,
        dtype=torch.float32
    ).unsqueeze(0)

    graph.smiles = smiles
    graph.drug_id = str(drug_id)
    graph.drug_name = str(drug_name)

    return graph


# ------------------------------------------------------------
# Build baseline molecular graph corpus
# ------------------------------------------------------------

baseline_graphs = []

for row in tqdm(
    chem_df.itertuples(index=False),
    total=len(chem_df),
    desc="Generating 3-D molecular graphs"
):

    graph = smiles_to_graph(
        row.smiles,
        drug_id=row.drug_id,
        drug_name=row.drug_name
    )

    if graph is not None:
        baseline_graphs.append(graph)


print("\nSuccessfully generated graphs:", len(baseline_graphs))

if len(baseline_graphs) < 30:
    raise RuntimeError(
        "Too few molecular graphs were generated. "
        "Check your DrugBank SMILES records."
    )

NODE_FEATURE_DIM = baseline_graphs[0].x.shape[1]
EDGE_FEATURE_DIM = baseline_graphs[0].edge_attr.shape[1]

print("Node feature dimension:", NODE_FEATURE_DIM)
print("Bond feature dimension:", EDGE_FEATURE_DIM)

RDKit fragment-based SA scorer loaded.
Canonical single-component molecules: 592


Generating 3-D molecular graphs:   0%|          | 0/592 [00:00<?, ?it/s]

[19:32:04] UFFTYPER: Unrecognized atom type: Ca1+2 (0)
[19:32:05] UFFTYPER: Unrecognized charge state for atom: 1
[19:32:05] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[19:32:05] UFFTYPER: Unrecognized charge state for atom: 0
[19:32:05] UFFTYPER: Unrecognized charge state for atom: 0
[19:32:05] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[19:32:06] UFFTYPER: Unrecognized atom type: Fe5+2 (17)
[19:33:04] UFFTYPER: Unrecognized atom type: Fe5+2 (17)
[19:35:26] UFFTYPER: Unrecognized hybridization for atom: 2
[19:35:26] UFFTYPER: Unrecognized atom type: Fe+2 (2)
[19:35:26] UFFTYPER: Unrecognized hybridization for atom: 2
[19:35:26] UFFTYPER: Unrecognized atom type: Fe+2 (2)



Successfully generated graphs: 579
Node feature dimension: 12
Bond feature dimension: 5
